# S3 n32 w256

condition=n32_random_points  width=tiny_256  act=relu  pool=max  epochs=2001  seed=0


In [ ]:
CONDITION = "n32_random_points"
HD, ACT, POOL = "tiny_256", "relu", "max"
EPOCHS, SEED = 2001, 0
DIM_K = 1024
CUSTOM_DIMS = [256, 128, 64, 32, 16]


In [ ]:
import os, sys, glob, shutil, time, json
import numpy as np
import torch
assert torch.cuda.is_available(), "No GPU - accelerator must be GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
_ = (torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")).sum().item()
print("CUDA OK")

INPUT = "/kaggle/input"
cands = glob.glob(os.path.join(INPUT, "**", "toy_datasets"), recursive=True)
if not cands:
    raise RuntimeError("toy_datasets not found; saw " + str([r for r,d,f in os.walk(INPUT)][:30]))
ROOT = os.path.dirname(cands[0])
WORK = "/kaggle/working/data"
os.makedirs(WORK, exist_ok=True)
for n in ["toy_datasets", "ground_truth"]:
    if not os.path.exists(os.path.join(WORK, n)):
        shutil.copytree(os.path.join(ROOT, n), os.path.join(WORK, n))
hits = glob.glob(os.path.join(INPUT, "**", "experiment_architecture_combined.py"), recursive=True)
shutil.copy(hits[0], "/kaggle/working/experiment_architecture_combined.py")
DATA = os.path.join(WORK, "toy_datasets", CONDITION)
GT = os.path.join(WORK, "ground_truth", CONDITION)
OUT = "/kaggle/working/run"
print("data:", DATA, "| samples:", len(glob.glob(DATA + "/*.npy")))


In [ ]:
# ---- EXPLICIT held-out split, asserted (no edit to the training script) ----
# get_sample() does sorted(os.listdir(DATA)); the loader is
# DataLoader(batch_size=2, shuffle=False, drop_last=True). Over 5 samples that
# yields batches [0,1] and [2,3] and silently drops index 4 EVERY epoch. We do
# not rely on that silently -- we rebuild the identical loader and prove it.
sys.path.insert(0, "/kaggle/working")
import experiment_architecture_combined as X

BATCH_SIZE = 2
files = [os.path.join(DATA, f) for f in sorted(os.listdir(DATA))]
assert len(files) == 5, "expected 5 samples, got %d" % len(files)
assert all(f.endswith(".npy") for f in files), "non-.npy entry in " + DATA

TRAIN_FILES, HELDOUT_FILE = files[:4], files[4]

_ds = X.load_dataset(DATA)
_dl = torch.utils.data.DataLoader(_ds, batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=0, drop_last=True)
seen = np.concatenate([b.numpy() for b in _dl], axis=0)
assert len(_ds) == 5, "dataset size changed"
assert seen.shape[0] == 4, "expected 4 seen samples, got %d" % seen.shape[0]

ref = np.stack([np.load(f).astype(np.float32) for f in TRAIN_FILES])
assert np.array_equal(seen, ref), "the 4 seen samples are NOT files 1-4"

held = np.load(HELDOUT_FILE).astype(np.float32)
assert not any(np.array_equal(held, s) for s in seen), "HELD-OUT SAMPLE LEAKED INTO TRAINING"

print("split OK")
print("  train  (seen)  :", [os.path.basename(f) for f in TRAIN_FILES])
print("  heldout(unseen):", os.path.basename(HELDOUT_FILE))
del _ds, _dl, seen, ref, held


In [ ]:
# Run in-process so the weight-init seed can be overridden without editing the
# training script (its torch.manual_seed(0) is hardcoded at line 578).
sys.path.insert(0, "/kaggle/working")
import experiment_architecture_combined as X

_orig_seed = torch.manual_seed
torch.manual_seed = lambda _ignored: _orig_seed(SEED)
print("weight-init seed overridden to", SEED)

# register a trunk size the script does not ship with (no file edit needed:
# parse_args reads HIDDEN_DIM_CONFIGS at call time, and we inject before main)
if CUSTOM_DIMS is not None:
    X.HIDDEN_DIM_CONFIGS[HD] = CUSTOM_DIMS
    print("registered", HD, "=", CUSTOM_DIMS)
print("dim_k =", DIM_K)

sys.argv = ["run", "--traindata-path", DATA, "--testdata-path", DATA,
            "--outdir", OUT, "--device", "cuda", "--epochs", str(EPOCHS),
            "--batch-size", "2", "--mode", "full_grid", "--dim-k", str(DIM_K),
            "--hidden-dims-subset", HD, "--activations-subset", ACT,
            "--poolings-subset", POOL,
            "--save-every", str(EPOCHS - 1)]
t0 = time.time()
X.main()
torch.manual_seed = _orig_seed
print("elapsed hours:", round((time.time() - t0) / 3600, 3))


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

run_dir = glob.glob(os.path.join(OUT, "hd=*"))[0]
hist = np.genfromtxt(os.path.join(run_dir, "history.csv"), delimiter=",",
                     names=True, dtype=None, encoding=None)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, k in enumerate(["total", "chamfer", "pinns"]):
    ax[i].plot(hist[k]); ax[i].set_yscale("log")
    ax[i].set_title(k); ax[i].set_xlabel("epoch"); ax[i].grid(alpha=.3)
plt.suptitle(os.path.basename(run_dir) + "  seed=" + str(SEED))
plt.tight_layout(); plt.savefig("/kaggle/working/loss_curves.png", dpi=110)
plt.show()

n = len(hist["total"])
print("epochs run:", n)
for frac in [0.25, 0.5, 0.75, 1.0]:
    i = int(n * frac) - 1
    print("  at %5d: total %.6f  chamfer %.6f  pinns %.6f"
          % (i + 1, hist["total"][i], hist["chamfer"][i], hist["pinns"][i]))
last10 = hist["total"][-max(1, n // 10):]
print("plateaued?  last-10%% relative spread = %.3e" % (last10.std() / abs(last10.mean())))


In [ ]:
def _gt(f):
    return os.path.join(GT, os.path.basename(f).replace(".npy", "_ground_truth.npy"))

model = X.PhysicsInformedRegistration(dim_k=DIM_K, hidden_dims=X.HIDDEN_DIM_CONFIGS[HD],
                                      shared_backbone=False, activation=ACT, pool_type=POOL).cuda()
model.load_state_dict(torch.load(os.path.join(run_dir, "model_last.pth"), map_location="cuda"))
model.eval()
nparams = sum(p.numel() for p in model.parameters())

def rmse_of(f):
    a = np.load(f).astype(np.float32)
    gt = np.load(_gt(f))
    b = torch.tensor(a).unsqueeze(0).cuda()
    pred = (model(b)["deformed_source"] - b[:, 0, :, :]).detach().cpu().numpy()[0]
    return float(np.sqrt(((pred - gt) ** 2).sum(1).mean()))

def zero_of(f):
    return float(np.sqrt((np.load(_gt(f)) ** 2).sum(1).mean()))

seen_rmse   = [rmse_of(f) for f in TRAIN_FILES]
unseen_rmse = rmse_of(HELDOUT_FILE)
seen_zero   = [zero_of(f) for f in TRAIN_FILES]
unseen_zero = zero_of(HELDOUT_FILE)

res = {"condition": CONDITION, "dim_k": DIM_K, "hidden_dims": HD, "activation": ACT,
       "pool": POOL, "backbone": "separate", "seed": SEED, "epochs": EPOCHS,
       "params_M": nparams / 1e6,
       "train_files": [os.path.basename(f) for f in TRAIN_FILES],
       "heldout_file": os.path.basename(HELDOUT_FILE),
       "rmse_seen_mean": float(np.mean(seen_rmse)),
       "rmse_seen_per_sample": seen_rmse,
       "rmse_unseen": unseen_rmse,
       "generalisation_gap": unseen_rmse - float(np.mean(seen_rmse)),
       "generalisation_ratio": unseen_rmse / float(np.mean(seen_rmse)),
       "rmse_predict_zero_seen": float(np.mean(seen_zero)),
       "rmse_predict_zero_unseen": unseen_zero,
       "rmse_mean_all5": float(np.mean(seen_rmse + [unseen_rmse])),
       "final_total": float(hist["total"][-1]),
       "final_chamfer": float(hist["chamfer"][-1]),
       "final_pinns": float(hist["pinns"][-1])}
with open("/kaggle/working/result.json", "w") as fh:
    json.dump(res, fh, indent=1)

print(json.dumps(res, indent=1))
print()
print("RMSE seen  (files 1-4) : %.6e   (predict-zero %.6e)" % (res["rmse_seen_mean"], res["rmse_predict_zero_seen"]))
print("RMSE unseen(file 5)    : %.6e   (predict-zero %.6e)" % (unseen_rmse, unseen_zero))
print("generalisation ratio   : %.3f" % res["generalisation_ratio"])
print("BEATS DOING NOTHING (unseen)? ", unseen_rmse < unseen_zero)
